# Pollution as the endogenous variable (MERRA-2)

The main specification instruments the district's own fire intensity (log FRP) with upwind fire. Because smoke from upwind fires could affect conflict directly, the exclusion restriction for that IV is hard to defend, and the paper treats the reduced form as primary. This notebook replaces fire intensity with a direct measure of **air pollution**, so the IV estimates the effect of pollution itself:

* **Surface PM2.5** (µg/m³) from MERRA-2 monthly aerosol diagnostics (M2TMNXAER), built from the standard formula: dust (< 2.5 µm) + sea salt (< 2.5 µm) + organic carbon + black carbon + 1.375 × sulfate (Buchard et al. 2016).
* **Aerosol optical depth** (AOD at 550 nm, `TOTEXTTAU`) as a second measure.

The exclusion restriction becomes: upwind fire affects conflict only through the pollution it brings. **Caveat:** MERRA-2 is a reanalysis. It assimilates satellite AOD, but its biomass-burning emissions come from QFED, which is built from MODIS fire detections, so its PM2.5 is partly derived from fire data of the kind the instrument uses.

Downloads use NASA Earthdata credentials in `~/.netrc` (no credentials are stored in this notebook). Each global monthly file (~84 MB) is downloaded, cropped to Indonesia and deleted; only the cropped file is kept.

In [1]:
import os
import re
import warnings
import subprocess
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import pyfixest as pf
from scipy import stats
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')

ROOT         = '/Users/hariaksha/Documents/GitHub/climate-conflict'
PANEL_CSV    = f'{ROOT}/analysis/iv_panel_district.csv'
DISTRICT_SHP = f'{ROOT}/data/administrative/gadm41_IDN_shp/gadm41_IDN_2.shp'
ACLED_XLSX   = f'{ROOT}/data/unrest/ACLED Data_2026-06-10.xlsx'
POLL_DIR     = f'{ROOT}/data/pollution'
RAW_DIR      = f'{POLL_DIR}/merra2_raw_tmp'
POLL_NC      = f'{POLL_DIR}/merra2_aer_indonesia_2015_2025.nc'
POLL_CSV     = f'{POLL_DIR}/merra2_pm25_district_monthly.csv'
OUT_CSV      = f'{ROOT}/analysis/output/pollution_iv_results.csv'

BASE_URL = 'https://goldsmr4.gesdisc.eosdis.nasa.gov/data/MERRA2_MONTHLY/M2TMNXAER.5.12.4'
VARS     = ['TOTEXTTAU', 'DUSMASS25', 'SSSMASS25', 'OCSMASS', 'BCSMASS', 'SO4SMASS']
LAT, LON = slice(-11, 6), slice(95, 141)
START_YEAR, END_YEAR, END_MONTH = 2015, 2025, 6

INSTR    = ['log_upwind_frp', 'log_upwind_frp_l1']
OUTCOMES = ['events', 'pv_events', 'riots_vac', 'vac']
FE       = 'district + year_month'
VCOV     = {'CRV1': 'province'}
os.makedirs(RAW_DIR, exist_ok=True)

## 1. Download and crop MERRA-2

File names depend on the MERRA-2 production stream (e.g. `MERRA2_400`, `MERRA2_401`), so each year's directory listing is read to find them. Skipped if the cropped file already exists.

In [2]:
def curl(args):
    # -n reads ~/.netrc; the cookie jar holds the Earthdata session across redirects
    cookie = f'{RAW_DIR}/.cookies'
    return subprocess.run(['curl', '-s', '-n', '-L', '-c', cookie, '-b', cookie] + args,
                          capture_output=True, check=True)

if not os.path.exists(POLL_NC):
    parts = []
    for year in range(START_YEAR, END_YEAR + 1):
        listing = curl([f'{BASE_URL}/{year}/']).stdout.decode()
        files = sorted(set(re.findall(r'MERRA2_\d+\.tavgM_2d_aer_Nx\.\d{6}\.nc4(?!\.)', listing)))
        for fn in files:
            ym = int(fn.split('.')[-2])
            if year == END_YEAR and ym % 100 > END_MONTH:
                continue
            path = f'{RAW_DIR}/{fn}'
            curl(['-o', path, f'{BASE_URL}/{year}/{fn}'])
            with xr.open_dataset(path) as ds:
                parts.append(ds[VARS].sel(lat=LAT, lon=LON).load())
            os.remove(path)
        print(f'{year}: {len(files)} files')
    xr.concat(parts, dim='time').to_netcdf(POLL_NC)
    print(f'Saved {len(parts)} months to {POLL_NC}')
else:
    print(f'{POLL_NC} already exists -- skipping download.')

ds = xr.open_dataset(POLL_NC)
print(ds)

/Users/hariaksha/Documents/GitHub/climate-conflict/data/pollution/merra2_aer_indonesia_2015_2025.nc already exists -- skipping download.


<xarray.Dataset> Size: 8MB
Dimensions:    (time: 126, lat: 35, lon: 74)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2015-01-01T00:30:00 ... 2025-06-01T0...
  * lat        (lat) float64 280B -11.0 -10.5 -10.0 -9.5 ... 4.5 5.0 5.5 6.0
  * lon        (lon) float64 592B 95.0 95.62 96.25 96.88 ... 139.4 140.0 140.6
Data variables:
    TOTEXTTAU  (time, lat, lon) float32 1MB ...
    DUSMASS25  (time, lat, lon) float32 1MB ...
    SSSMASS25  (time, lat, lon) float32 1MB ...
    OCSMASS    (time, lat, lon) float32 1MB ...
    BCSMASS    (time, lat, lon) float32 1MB ...
    SO4SMASS   (time, lat, lon) float32 1MB ...
Attributes: (12/30)
    History:                           Original file generated: Mon Mar 23 19...
    Filename:                          MERRA2_400.tavgM_2d_aer_Nx.201501.nc4
    Comment:                           GMAO filename: d5124_m2_jan10.tavg1_2d...
    Conventions:                       CF-1
    Institution:                       NASA Global Modeling and Assimil

## 2. Aggregate to district-months

The MERRA-2 grid is coarse (0.5° × 0.625°, roughly 55 × 70 km), so many districts contain no grid-cell centre. Each district is assigned the mean of the cells whose centres fall inside it; districts containing none use the cell nearest their representative point (as for ERA5 in `weather-controls.ipynb`). Mass concentrations are converted from kg/m³ to µg/m³.

In [3]:
districts_gdf = gpd.read_file(DISTRICT_SHP)[['NAME_1', 'NAME_2', 'geometry']].rename(
    columns={'NAME_1': 'province', 'NAME_2': 'district'})
dup = districts_gdf['district'].duplicated(keep=False)
districts_gdf.loc[dup, 'district'] = (districts_gdf.loc[dup, 'district'] + ' ('
                                      + districts_gdf.loc[dup, 'province'] + ')')
rep = districts_gdf.geometry.representative_point()
districts_gdf['dist_lat'], districts_gdf['dist_lon'] = rep.y, rep.x

px = ds.to_dataframe().reset_index()
px['time'] = pd.to_datetime(px['time'])
px['year'], px['month'] = px['time'].dt.year, px['time'].dt.month
px['pm25'] = (px['DUSMASS25'] + px['SSSMASS25'] + px['OCSMASS'] + px['BCSMASS']
              + px['SO4SMASS'] * 132.14 / 96.06) * 1e9
px['aod'] = px['TOTEXTTAU']

grid = px[['lat', 'lon']].drop_duplicates().reset_index(drop=True)
gg = gpd.GeoDataFrame(grid, geometry=gpd.points_from_xy(grid['lon'], grid['lat']), crs='EPSG:4326')
joined = gpd.sjoin(gg, districts_gdf[['district', 'geometry']], how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]
grid['district'] = joined['district'].values
missing = districts_gdf[~districts_gdf['district'].isin(set(grid['district'].dropna()))]
_, nn = cKDTree(grid[['lat', 'lon']].values).query(missing[['dist_lat', 'dist_lon']].values)
fallback = grid.iloc[nn][['lat', 'lon']].reset_index(drop=True)
fallback['district'] = missing['district'].values
grid = pd.concat([grid.dropna(subset=['district']), fallback], ignore_index=True)
print(f'Districts with pollution data: {grid["district"].nunique()} ({len(missing)} via nearest-cell fallback)')

poll = (px.merge(grid, on=['lat', 'lon'])
          .groupby(['district', 'year', 'month'])[['pm25', 'aod']].mean().reset_index())
poll.to_csv(POLL_CSV, index=False)
print(f'Saved: {POLL_CSV}')
print(poll[['pm25', 'aod']].describe().round(2))
print('\nMean PM2.5 by month (haze season is Aug-Oct):')
print(poll.groupby('month')['pm25'].mean().round(1).to_dict())

Districts with pollution data: 502 (243 via nearest-cell fallback)
Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/data/pollution/merra2_pm25_district_monthly.csv
           pm25       aod
count  63252.00  63252.00
mean      11.36      0.17
std        7.56      0.12
min        1.04      0.02
25%        6.91      0.10
50%        9.64      0.14
75%       14.24      0.22
max      239.07      1.85

Mean PM2.5 by month (haze season is Aug-Oct):
{1: 9.300000190734863, 2: 10.399999618530273, 3: 10.300000190734863, 4: 10.5, 5: 10.800000190734863, 6: 10.899999618530273, 7: 11.5, 8: 12.399999618530273, 9: 15.199999809265137, 10: 15.199999809265137, 11: 11.100000381469727, 12: 9.300000190734863}


## 3. Merge into the IV panel

Pollution enters in logs, with a one-month lag, matching the fire variables.

In [4]:
panel = pd.read_csv(PANEL_CSV)
poll = poll.sort_values(['district', 'year', 'month'])
for v in ['pm25', 'aod']:
    poll[f'log_{v}'] = np.log(poll[v])
    poll[f'log_{v}_l1'] = poll.groupby('district')[f'log_{v}'].shift(1)
panel = panel.merge(poll, on=['district', 'year', 'month'], how='left')
panel = panel.dropna(subset=['log_pm25', 'log_pm25_l1']).reset_index(drop=True)
print(f'Panel: {len(panel):,} obs, {panel["district"].nunique()} districts')
print(f'Corr(log upwind FRP, log PM2.5): {panel["log_upwind_frp"].corr(panel["log_pm25"]):.3f}')
print(f'Corr(log own FRP,    log PM2.5): {panel["log_frp"].corr(panel["log_pm25"]):.3f}')

Panel: 60,125 obs, 481 districts
Corr(log upwind FRP, log PM2.5): 0.249
Corr(log own FRP,    log PM2.5): 0.076


## 4. Event-type outcomes

As in `weather-controls.ipynb`, Section 4.

In [5]:
districts_gdf = gpd.read_file(DISTRICT_SHP)[['NAME_1', 'NAME_2', 'geometry']].rename(
    columns={'NAME_1': 'province', 'NAME_2': 'district'})
dup = districts_gdf['district'].duplicated(keep=False)
districts_gdf.loc[dup, 'district'] = (districts_gdf.loc[dup, 'district'] + ' ('
                                      + districts_gdf.loc[dup, 'province'] + ')')

acled = pd.read_excel(ACLED_XLSX, usecols=['event_date', 'event_type', 'latitude', 'longitude'])
acled['event_date'] = pd.to_datetime(acled['event_date'])
acled['year'], acled['month'] = acled['event_date'].dt.year, acled['event_date'].dt.month
ag = gpd.GeoDataFrame(acled, geometry=gpd.points_from_xy(acled['longitude'], acled['latitude']), crs='EPSG:4326')
j = gpd.sjoin(ag, districts_gdf[['district', 'geometry']], how='left', predicate='within')
j = j[~j.index.duplicated(keep='first')]
acled['district'] = j['district'].reindex(acled.index).values
off = acled['district'].isna()
near = gpd.sjoin_nearest(ag.loc[off].to_crs(3395), districts_gdf[['district', 'geometry']].to_crs(3395),
                         how='left', max_distance=20_000)
near = near[~near.index.duplicated(keep='first')]
acled.loc[off, 'district'] = near['district'].reindex(acled.index[off]).values

acled['riots_vac'] = acled['event_type'].isin(['Riots', 'Violence against civilians']).astype(int)
acled['vac'] = (acled['event_type'] == 'Violence against civilians').astype(int)
counts = acled.groupby(['district', 'year', 'month'])[['riots_vac', 'vac']].sum().reset_index()
panel = panel.merge(counts, on=['district', 'year', 'month'], how='left').fillna({'riots_vac': 0, 'vac': 0})
print(panel[OUTCOMES].sum())

events       21653.0
pv_events     3520.0
riots_vac     2824.0
vac           1067.0
dtype: float64


## 5. Estimates

For each pollution measure and sample: the first stage (log pollution on contemporaneous and lagged log upwind FRP, with lagged log pollution as a control), the effective F, the 2SLS coefficient on instrumented log pollution, and the Anderson–Rubin p-value. A second version also controls for the district's own fire (log FRP and its lag), so the pollution coefficient is not picking up a direct effect of local burning. Samples: full panel, conflict-active (≥ 30% of months), and conflict-active restricted to 2018–2025.

In [6]:
def effective_f(data, instruments, controls=(), fe=FE, endog='log_frp'):
    data = data.reset_index(drop=True)
    ctrl = ' + '.join(controls)
    rhs = ' + '.join(list(instruments) + list(controls))
    fs = pf.feols(f'{endog} ~ {rhs} | {fe}', data=data, vcov=VCOV)
    Z = np.column_stack([pf.feols(f'{z} ~ {ctrl if ctrl else "1"} | {fe}', data=data).resid()
                         for z in instruments])
    names = list(fs.coef().index)
    idx = [names.index(z) for z in instruments]
    pi = fs.coef()[list(instruments)].to_numpy()
    V = fs._vcov[np.ix_(idx, idx)]
    S = Z.T @ Z
    return float(pi @ S @ pi / np.trace(V @ S))

def wald_p(m, terms, G):
    names = list(m.coef().index)
    idx = [names.index(t) for t in terms]
    b = m.coef()[terms].to_numpy()
    V = m._vcov[np.ix_(idx, idx)]
    W = float(b @ np.linalg.solve(V, b))
    return stats.f.sf(W / len(terms), len(terms), G - 1)

def ar_p(data, y, controls, fe=FE):
    m = pf.feols(f'{y} ~ {" + ".join(INSTR + list(controls))} | {fe}', data=data, vcov=VCOV)
    return wald_p(m, INSTR, data['province'].nunique())


share = panel.groupby('district')['events'].apply(lambda s: (s > 0).mean())
active = share[share >= 0.30].index
samples = {
    'full panel': panel,
    'active >= 30%': panel[panel['district'].isin(active)],
    'active >= 30%, 2018+': panel[panel['district'].isin(active) & (panel['year'] >= 2018)],
}
print(f'Active >= 30% districts: {len(active)}')

rows = []
for endog in ['log_pm25', 'log_aod']:
    for ctrl_label, extra in [('baseline', []), ('+ own fire', ['log_frp', 'log_frp_l1'])]:
        ctrls = [f'{endog}_l1'] + extra
        for sample, d in samples.items():
            d = d.dropna(subset=ctrls).reset_index(drop=True)
            effF = effective_f(d, INSTR, ctrls, endog=endog)
            fs = pf.feols(f'{endog} ~ {" + ".join(INSTR + ctrls)} | {FE}', data=d, vcov=VCOV).tidy()
            print(f'\n== {endog} | {ctrl_label} | {sample} | effective F = {effF:.1f} | '
                  f'first stage {fs.loc["log_upwind_frp", "Estimate"]:+.4f} (p={fs.loc["log_upwind_frp", "Pr(>|t|)"]:.3f}) ==')
            for y in OUTCOMES:
                iv = pf.feols(f'{y} ~ {" + ".join(ctrls)} | {FE} | {endog} ~ {" + ".join(INSTR)}',
                              data=d, vcov=VCOV).tidy().loc[endog]
                arp = ar_p(d, y, ctrls)
                rows.append({'endog': endog, 'controls': ctrl_label, 'sample': sample, 'outcome': y,
                             'effective_F': effF, 'first_stage': fs.loc['log_upwind_frp', 'Estimate'],
                             'iv_coef': iv['Estimate'], 'iv_se': iv['Std. Error'], 'iv_p': iv['Pr(>|t|)'],
                             'ar_p': arp, 'mean_y': d[y].mean()})
                print(f"  {y:10s} IV {iv['Estimate']:+.4f} (se {iv['Std. Error']:.4f}, p={iv['Pr(>|t|)']:.3f}) | AR p={arp:.3f}")

res = pd.DataFrame(rows)
res.to_csv(OUT_CSV, index=False)
print(f'\nSaved: {OUT_CSV}')

Active >= 30% districts: 85



== log_pm25 | baseline | full panel | effective F = 41.1 | first stage +0.0275 (p=0.000) ==


  events     IV -0.0792 (se 0.0760, p=0.304) | AR p=0.074


  pv_events  IV -0.0229 (se 0.0207, p=0.277) | AR p=0.021


  riots_vac  IV -0.0255 (se 0.0214, p=0.242) | AR p=0.027


  vac        IV -0.0232 (se 0.0156, p=0.147) | AR p=0.058

== log_pm25 | baseline | active >= 30% | effective F = 25.7 | first stage +0.0212 (p=0.000) ==
  events     IV -1.2271 (se 0.4056, p=0.005) | AR p=0.004
  pv_events  IV -0.3391 (se 0.1751, p=0.062) | AR p=0.154


... (19 outputs left out)

  riots_vac  IV -0.0205 (se 0.0403, p=0.614) | AR p=0.021


  vac        IV -0.0324 (se 0.0293, p=0.276) | AR p=0.083

== log_aod | + own fire | active >= 30% | effective F = 14.3 | first stage +0.0147 (p=0.000) ==
  events     IV -1.6575 (se 0.8428, p=0.058) | AR p=0.033
  pv_events  IV -0.4443 (se 0.3672, p=0.235) | AR p=0.160


  riots_vac  IV -0.4513 (se 0.3305, p=0.182) | AR p=0.077
  vac        IV -0.3148 (se 0.2472, p=0.212) | AR p=0.027

== log_aod | + own fire | active >= 30%, 2018+ | effective F = 10.1 | first stage +0.0147 (p=0.000) ==
  events     IV -2.1058 (se 1.0796, p=0.060) | AR p=0.038


  pv_events  IV -0.6810 (se 0.5775, p=0.247) | AR p=0.077
  riots_vac  IV -0.5691 (se 0.4463, p=0.211) | AR p=0.040
  vac        IV -0.4066 (se 0.3339, p=0.232) | AR p=0.009

Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/analysis/output/pollution_iv_results.csv


## 6. Interpretation of magnitudes

The IV coefficient is the change in monthly events per one-unit change in log PM2.5, i.e. per roughly 2.7-fold increase. Below, it is rescaled to a 10% increase in PM2.5 and expressed relative to the sample mean of each outcome (baseline controls).

In [7]:
s = res[(res['endog'] == 'log_pm25') & (res['controls'] == 'baseline')].copy()
s['effect_10pct'] = s['iv_coef'] * np.log(1.1)
s['effect_10pct_rel_mean'] = s['effect_10pct'] / s['mean_y']
print(s[['sample', 'outcome', 'effective_F', 'iv_coef', 'iv_p', 'ar_p', 'mean_y',
         'effect_10pct', 'effect_10pct_rel_mean']].round(4).to_string(index=False))

              sample   outcome  effective_F  iv_coef   iv_p   ar_p  mean_y  effect_10pct  effect_10pct_rel_mean
          full panel    events      41.0728  -0.0792 0.3044 0.0735  0.3601       -0.0076                -0.0210
          full panel pv_events      41.0728  -0.0229 0.2766 0.0207  0.0585       -0.0022                -0.0373
          full panel riots_vac      41.0728  -0.0255 0.2424 0.0268  0.0470       -0.0024                -0.0518
          full panel       vac      41.0728  -0.0232 0.1472 0.0584  0.0177       -0.0022                -0.1244
       active >= 30%    events      25.6856  -1.2271 0.0049 0.0037  1.3975       -0.1170                -0.0837
       active >= 30% pv_events      25.6856  -0.3391 0.0618 0.1535  0.2176       -0.0323                -0.1485
       active >= 30% riots_vac      25.6856  -0.3723 0.0317 0.0863  0.1666       -0.0355                -0.2130
       active >= 30%       vac      25.6856  -0.2575 0.0507 0.0361  0.0719       -0.0245                

## 7. Districts that contain their own MERRA-2 grid point

243 of 502 districts contain no MERRA-2 grid-cell centre and are assigned the cell nearest their representative point, so their pollution is measured with extra error (the cell can lie in a neighbouring district or over the sea). Classical measurement error in the endogenous variable does not bias 2SLS, but it weakens the first stage; non-classical error (e.g. the nearest cell systematically lying upwind or offshore) could. This section re-estimates the pollution IV only on districts whose pollution comes from at least one grid point inside their own boundary.

In [8]:
own_grid = set(grid.loc[~grid['district'].isin(missing['district']), 'district'])
print(f'Districts containing their own grid point: {len(own_grid)} of {grid["district"].nunique()}')

og_samples = {}
for sample, d in samples.items():
    og_samples[sample + ', own grid point'] = d[d['district'].isin(own_grid)]
    print(f'  {sample:22s}: {d["district"].nunique():3d} -> {d[d["district"].isin(own_grid)]["district"].nunique():3d} districts')

og_rows = []
for endog in ['log_pm25', 'log_aod']:
    for ctrl_label, extra in [('baseline', []), ('+ own fire', ['log_frp', 'log_frp_l1'])]:
        ctrls = [f'{endog}_l1'] + extra
        for sample, d in og_samples.items():
            d = d.dropna(subset=ctrls).reset_index(drop=True)
            effF = effective_f(d, INSTR, ctrls, endog=endog)
            fs = pf.feols(f'{endog} ~ {" + ".join(INSTR + ctrls)} | {FE}', data=d, vcov=VCOV).tidy()
            print(f'\n== {endog} | {ctrl_label} | {sample} | effective F = {effF:.1f} | '
                  f'first stage {fs.loc["log_upwind_frp", "Estimate"]:+.4f} (p={fs.loc["log_upwind_frp", "Pr(>|t|)"]:.3f}) ==')
            for y in OUTCOMES:
                iv = pf.feols(f'{y} ~ {" + ".join(ctrls)} | {FE} | {endog} ~ {" + ".join(INSTR)}',
                              data=d, vcov=VCOV).tidy().loc[endog]
                arp = ar_p(d, y, ctrls)
                og_rows.append({'endog': endog, 'controls': ctrl_label, 'sample': sample, 'outcome': y,
                                'effective_F': effF, 'first_stage': fs.loc['log_upwind_frp', 'Estimate'],
                                'iv_coef': iv['Estimate'], 'iv_se': iv['Std. Error'], 'iv_p': iv['Pr(>|t|)'],
                                'ar_p': arp, 'mean_y': d[y].mean(), 'n_districts': d['district'].nunique()})
                print(f"  {y:10s} IV {iv['Estimate']:+.4f} (se {iv['Std. Error']:.4f}, p={iv['Pr(>|t|)']:.3f}) | AR p={arp:.3f}")

res = pd.concat([res, pd.DataFrame(og_rows)], ignore_index=True)
res.to_csv(OUT_CSV, index=False)
print(f'\nSaved: {OUT_CSV} (now includes own-grid-point rows)')

Districts containing their own grid point: 259 of 502
  full panel            : 481 -> 249 districts
  active >= 30%         :  85 ->  25 districts
  active >= 30%, 2018+  :  85 ->  25 districts



== log_pm25 | baseline | full panel, own grid point | effective F = 49.0 | first stage +0.0340 (p=0.000) ==


  events     IV +0.0378 (se 0.0585, p=0.523) | AR p=0.154


  pv_events  IV +0.0147 (se 0.0293, p=0.619) | AR p=0.027


  riots_vac  IV +0.0160 (se 0.0252, p=0.532) | AR p=0.017


  vac        IV -0.0117 (se 0.0181, p=0.524) | AR p=0.103

== log_pm25 | baseline | active >= 30%, own grid point | effective F = 6.7 | first stage +0.0216 (p=0.029) ==


... (11 outputs left out)


== log_aod | + own fire | full panel, own grid point | effective F = 37.8 | first stage +0.0196 (p=0.000) ==
  events     IV +0.0044 (se 0.1344, p=0.974) | AR p=0.180


  pv_events  IV -0.0016 (se 0.0746, p=0.983) | AR p=0.123
  riots_vac  IV +0.0078 (se 0.0618, p=0.901) | AR p=0.116


  vac        IV -0.0234 (se 0.0401, p=0.565) | AR p=0.213

== log_aod | + own fire | active >= 30%, own grid point | effective F = 5.8 | first stage +0.0150 (p=0.030) ==
  events     IV -0.9022 (se 1.8478, p=0.638) | AR p=0.290
  pv_events  IV -1.1554 (se 1.2294, p=0.375) | AR p=0.342
  riots_vac  IV -1.0835 (se 1.0590, p=0.336) | AR p=0.353
  vac        IV -0.7350 (se 0.7404, p=0.350) | AR p=0.252



== log_aod | + own fire | active >= 30%, 2018+, own grid point | effective F = 4.6 | first stage +0.0154 (p=0.053) ==
  events     IV -0.8797 (se 2.6405, p=0.748) | AR p=0.696
  pv_events  IV -1.2516 (se 1.6176, p=0.461) | AR p=0.625
  riots_vac  IV -1.2661 (se 1.3880, p=0.388) | AR p=0.289
  vac        IV -0.9174 (se 0.9264, p=0.351) | AR p=0.134

Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/analysis/output/pollution_iv_results.csv (now includes own-grid-point rows)


### 7b. Is the loss of precision about the pollution measure or the sample?

The own-grid-point restriction removes most conflict-active districts, which are disproportionately small urban districts (*kota*) that contain no grid-cell centre. The reduced form (conflict on upwind FRP) does not use MERRA-2 at all, so estimating it on the same subsamples separates the two explanations: if the reduced form is also imprecise among own-grid-point districts, the problem is sample size and composition, not the pollution measure.

In [9]:
d_act = samples['active >= 30%']
kota = d_act.groupby('district').size().index.to_series().str.startswith('Kota')
print(f'Conflict-active districts that are cities (Kota): {kota.sum()} of {kota.size}; '
      f'with own grid point: {sum(1 for x in kota.index if x in own_grid and x.startswith("Kota"))}')
rf_rows = []
for label, d in [('active >= 30%, all', d_act),
                 ('active >= 30%, own grid point', d_act[d_act['district'].isin(own_grid)]),
                 ('active >= 30%, nearest-cell fallback', d_act[~d_act['district'].isin(own_grid)]),
                 ('full panel, own grid point', samples['full panel'][samples['full panel']['district'].isin(own_grid)])]:
    print(f'\n{label} ({d["district"].nunique()} districts, mean events {d["events"].mean():.2f})')
    for y in OUTCOMES:
        m = pf.feols(f'{y} ~ {" + ".join(INSTR)} | {FE}', data=d, vcov=VCOV)
        t = m.tidy().loc['log_upwind_frp']
        jp = wald_p(m, INSTR, d['province'].nunique())
        rf_rows.append({'sample': label, 'outcome': y, 'rf_coef': t['Estimate'], 'rf_p': t['Pr(>|t|)'],
                        'joint_p': jp, 'n_districts': d['district'].nunique()})
        print(f"  {y:10s} RF {t['Estimate']:+.4f} (p={t['Pr(>|t|)']:.3f}) | joint p={jp:.3f}")
pd.DataFrame(rf_rows).to_csv(f'{ROOT}/analysis/output/pollution_owngrid_reduced_form.csv', index=False)
print('\nSaved: analysis/output/pollution_owngrid_reduced_form.csv')

Conflict-active districts that are cities (Kota): 18 of 85; with own grid point: 0

active >= 30%, all (85 districts, mean events 1.40)
  events     RF -0.0261 (p=0.002) | joint p=0.005


  pv_events  RF -0.0072 (p=0.078) | joint p=0.175
  riots_vac  RF -0.0079 (p=0.039) | joint p=0.086
  vac        RF -0.0055 (p=0.068) | joint p=0.026

active >= 30%, own grid point (25 districts, mean events 1.14)
  events     RF -0.0114 (p=0.578) | joint p=0.252
  pv_events  RF -0.0128 (p=0.258) | joint p=0.230
  riots_vac  RF -0.0124 (p=0.149) | joint p=0.324
  vac        RF -0.0087 (p=0.176) | joint p=0.360

active >= 30%, nearest-cell fallback (60 districts, mean events 1.50)
  events     RF -0.0295 (p=0.022) | joint p=0.062


  pv_events  RF -0.0044 (p=0.121) | joint p=0.293
  riots_vac  RF -0.0055 (p=0.073) | joint p=0.186


  vac        RF -0.0046 (p=0.107) | joint p=0.132

full panel, own grid point (249 districts, mean events 0.23)
  events     RF +0.0010 (p=0.617) | joint p=0.182
  pv_events  RF +0.0003 (p=0.760) | joint p=0.042
  riots_vac  RF +0.0004 (p=0.698) | joint p=0.046


  vac        RF -0.0005 (p=0.441) | joint p=0.252

Saved: analysis/output/pollution_owngrid_reduced_form.csv


## 8. Satellite aerosol optical depth (MODIS MAIAC, 5 km)

MERRA-2 is coarse and partly modelled. As an observed alternative, this section uses **MODIS MAIAC aerosol optical depth at 550 nm** from the daily 0.05° (about 5 km) climate-modelling-grid product **MCD19A2CMG v061** (Terra + Aqua), distributed by NASA LP DAAC. At 5 km, almost every district, including small cities, contains its own pixels.

* AOD is a column measure (ground to top of atmosphere), not surface concentration.
* Retrievals are missing under cloud, and very thick smoke can be flagged as cloud, so monthly means are averages over valid pixel-days. Coverage (the share of pixel-days with a valid retrieval) is recorded for each district-month.
* Each daily global file (~63 MB) is downloaded, cropped to Indonesia and deleted. Progress is cached month by month in `data/pollution/maiac_monthly_cache/` (sum and count of valid AOD per pixel), so an interrupted run resumes where it stopped.

Uses the same NASA Earthdata credentials in `~/.netrc`; requires the `pyhdf` package.

In [10]:
import json
import time
import threading
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pyhdf.SD import SD

MAIAC_CONCEPT = 'C2565807733-LPCLOUD'   # MCD19A2CMG v061 in NASA CMR
CACHE_DIR = f'{POLL_DIR}/maiac_monthly_cache'
TMP_DIR   = f'{POLL_DIR}/maiac_tmp'
MAIAC_CSV = f'{POLL_DIR}/maiac_aod_district_monthly.csv'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# 0.05-degree global grid, row 0 at 90N, column 0 at 180W; crop to Indonesia (6N-11S, 95E-141E)
R0, R1 = int((90 - 6) / 0.05), int((90 + 11) / 0.05)
C0, C1 = int((95 + 180) / 0.05), int((141 + 180) / 0.05)
m_lat = 90 - (np.arange(R0, R1) + 0.5) * 0.05
m_lon = -180 + (np.arange(C0, C1) + 0.5) * 0.05
FILL, SCALE = -28672, 0.001

def granule_urls(year, month):
    start = f'{year}-{month:02d}-01T00:00:00Z'
    end = (pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%dT23:59:59Z')
    q = (f'https://cmr.earthdata.nasa.gov/search/granules.json?collection_concept_id={MAIAC_CONCEPT}'
         f'&temporal={start},{end}&page_size=100')
    # curl uses the system certificate store (Python.org builds may lack one)
    feed = json.loads(subprocess.run(['curl', '-s', '-f', q], capture_output=True, check=True).stdout)['feed']['entry']
    urls = []
    for e in feed:
        urls += [l['href'] for l in e['links'] if l['href'].startswith('https') and l['href'].endswith('.hdf')]
    return sorted(set(urls))

def fetch_crop(url):
    # download one daily global file, return the cropped AOD array (NaN where missing), delete the file
    fn = f'{TMP_DIR}/{os.path.basename(url)}'
    cookie = f'{TMP_DIR}/.cookies_{threading.get_ident()}'
    for attempt in range(4):
        r = subprocess.run(['curl', '-s', '-n', '-L', '-c', cookie, '-b', cookie, '-o', fn,
                            '-w', '%{http_code}', url], capture_output=True)
        if r.stdout.decode() == '200':
            try:
                a = SD(fn).select('AOD_055')[R0:R1, C0:C1].astype(np.float32)
                os.remove(fn)
                a[a == FILL] = np.nan
                return a * SCALE
            except Exception:
                pass
        time.sleep(5 * (attempt + 1))
    if os.path.exists(fn):
        os.remove(fn)
    raise RuntimeError(f'failed: {url}')

months = [(y, m) for y in range(START_YEAR, END_YEAR + 1) for m in range(1, 13)
          if not (y == END_YEAR and m > END_MONTH)]
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    for y, m in months:
        out = f'{CACHE_DIR}/{y}{m:02d}.npz'
        if os.path.exists(out):
            continue
        urls = granule_urls(y, m)
        s = np.zeros((R1 - R0, C1 - C0), np.float32)
        n = np.zeros((R1 - R0, C1 - C0), np.int16)
        for a in pool.map(fetch_crop, urls):
            ok = np.isfinite(a)
            s[ok] += a[ok]
            n += ok
        np.savez_compressed(out, s=s, n=n, days=len(urls))
        print(f'{y}-{m:02d}: {len(urls)} days, valid pixel-day share {n.sum() / (n.size * max(len(urls), 1)):.2f} '
              f'({(time.time() - t0) / 60:.0f} min elapsed)', flush=True)
print(f'All {len(months)} months cached in {CACHE_DIR}')

All 126 months cached in /Users/hariaksha/Documents/GitHub/climate-conflict/data/pollution/maiac_monthly_cache


### 8a. Aggregate MAIAC to district-months

District AOD is the mean over all valid pixel-days whose pixel centre lies inside the district (total sum / total count). Districts with no pixel centre inside them (very small districts) use the nearest pixel. District-months with no valid retrieval are left missing.

In [11]:
# districts_gdf is rebuilt in Section 4 without representative points, so recompute them
rep = districts_gdf.geometry.representative_point()
districts_gdf['dist_lat'], districts_gdf['dist_lon'] = rep.y, rep.x

LL = np.array(np.meshgrid(m_lat, m_lon, indexing='ij')).reshape(2, -1).T
pix = gpd.GeoDataFrame({'pid': np.arange(len(LL))},
                       geometry=gpd.points_from_xy(LL[:, 1], LL[:, 0]), crs='EPSG:4326')
jj = gpd.sjoin(pix, districts_gdf[['district', 'geometry']], how='inner', predicate='within')
jj = jj[~jj.index.duplicated(keep='first')]
pix_map = jj[['pid', 'district']]
miss = districts_gdf[~districts_gdf['district'].isin(set(pix_map['district']))]
_, nn = cKDTree(LL).query(miss[['dist_lat', 'dist_lon']].values)
pix_map = pd.concat([pix_map, pd.DataFrame({'pid': nn, 'district': miss['district'].values})], ignore_index=True)
print(f'Districts with MAIAC pixels: {pix_map["district"].nunique()} ({len(miss)} via nearest pixel); '
      f'median pixels per district: {pix_map.groupby("district").size().median():.0f}')

pid = pix_map['pid'].to_numpy()
codes, dnames = pd.factorize(pix_map['district'])
recs = []
for y, m in months:
    z = np.load(f'{CACHE_DIR}/{y}{m:02d}.npz')
    s, n, days = z['s'].ravel()[pid], z['n'].ravel()[pid].astype(float), int(z['days'])
    S = np.bincount(codes, weights=s, minlength=len(dnames))
    N = np.bincount(codes, weights=n, minlength=len(dnames))
    P = np.bincount(codes, minlength=len(dnames))
    recs.append(pd.DataFrame({'district': dnames, 'year': y, 'month': m,
                              'maiac_aod': np.where(N > 0, S / np.maximum(N, 1), np.nan),
                              'maiac_coverage': N / np.maximum(P * max(days, 1), 1),
                              'maiac_days': days}))
maiac = pd.concat(recs, ignore_index=True)
maiac.to_csv(MAIAC_CSV, index=False)
print(f'Saved: {MAIAC_CSV}')
print(maiac[['maiac_aod', 'maiac_coverage']].describe().round(3))
print('\nMean coverage by calendar month:', maiac.groupby('month')['maiac_coverage'].mean().round(2).to_dict())
print('Mean AOD by calendar month:     ', maiac.groupby('month')['maiac_aod'].mean().round(2).to_dict())
mm = poll.merge(maiac, on=['district', 'year', 'month'])
print(f'\nCorr(log MAIAC AOD, log MERRA-2 AOD):   {np.log(mm["maiac_aod"]).corr(mm["log_aod"]):.3f}')
print(f'Corr(log MAIAC AOD, log MERRA-2 PM2.5): {np.log(mm["maiac_aod"]).corr(mm["log_pm25"]):.3f}')

Districts with MAIAC pixels: 502 (4 via nearest pixel); median pixels per district: 62


Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/data/pollution/maiac_aod_district_monthly.csv
       maiac_aod  maiac_coverage
count  63122.000       63252.000
mean       0.196           0.244
std        0.154           0.184
min        0.016           0.000
25%        0.140           0.112
50%        0.174           0.192
75%        0.221           0.320
max        4.361           1.000

Mean coverage by calendar month: {1: 0.12, 2: 0.15, 3: 0.19, 4: 0.24, 5: 0.3, 6: 0.32, 7: 0.36, 8: 0.35, 9: 0.33, 10: 0.26, 11: 0.18, 12: 0.13}
Mean AOD by calendar month:      {1: 0.19, 2: 0.19, 3: 0.18, 4: 0.18, 5: 0.18, 6: 0.17, 7: 0.17, 8: 0.18, 9: 0.25, 10: 0.28, 11: 0.21, 12: 0.19}

Corr(log MAIAC AOD, log MERRA-2 AOD):   0.703
Corr(log MAIAC AOD, log MERRA-2 PM2.5): 0.636


### 8b. MAIAC AOD as the endogenous variable

Same specification as Section 5, with log MAIAC AOD (and its lag) in place of MERRA-2 PM2.5. District-months with no valid retrieval, or whose previous month has none, drop out. The last block restricts to the 25 conflict-active districts that were excluded from the own-grid-point check in Section 7 because they contain no MERRA-2 grid point, to see whether the effect is present there once pollution is measured locally.

In [12]:
ms = maiac.sort_values(['district', 'year', 'month']).copy()
ms['log_maiac'] = np.log(ms['maiac_aod'].clip(lower=0.005))
ms['log_maiac_l1'] = ms.groupby('district')['log_maiac'].shift(1)
pm = panel.merge(ms[['district', 'year', 'month', 'log_maiac', 'log_maiac_l1', 'maiac_coverage']],
                 on=['district', 'year', 'month'], how='left')
pm = pm.dropna(subset=['log_maiac', 'log_maiac_l1']).reset_index(drop=True)
print(f'MAIAC panel: {len(pm):,} of {len(panel):,} district-months have valid AOD this month and last')
print(f'Corr(log upwind FRP, log MAIAC AOD): {pm["log_upwind_frp"].corr(pm["log_maiac"]):.3f}')

m_samples = {
    'full panel': pm,
    'active >= 30%': pm[pm['district'].isin(active)],
    'active >= 30%, 2018+': pm[pm['district'].isin(active) & (pm['year'] >= 2018)],
    'active >= 30%, no MERRA-2 grid point': pm[pm['district'].isin(active) & ~pm['district'].isin(own_grid)],
}
maiac_rows = []
for ctrl_label, extra in [('baseline', []), ('+ own fire', ['log_frp', 'log_frp_l1'])]:
    ctrls = ['log_maiac_l1'] + extra
    for sample, d in m_samples.items():
        d = d.dropna(subset=ctrls).reset_index(drop=True)
        effF = effective_f(d, INSTR, ctrls, endog='log_maiac')
        fs = pf.feols(f'log_maiac ~ {" + ".join(INSTR + ctrls)} | {FE}', data=d, vcov=VCOV).tidy()
        print(f'\n== log_maiac | {ctrl_label} | {sample} ({d["district"].nunique()} districts) | effective F = {effF:.1f} | '
              f'first stage {fs.loc["log_upwind_frp", "Estimate"]:+.4f} (p={fs.loc["log_upwind_frp", "Pr(>|t|)"]:.3f}) ==')
        for y in OUTCOMES:
            iv = pf.feols(f'{y} ~ {" + ".join(ctrls)} | {FE} | log_maiac ~ {" + ".join(INSTR)}',
                          data=d, vcov=VCOV).tidy().loc['log_maiac']
            arp = ar_p(d, y, ctrls)
            maiac_rows.append({'endog': 'log_maiac', 'controls': ctrl_label, 'sample': sample, 'outcome': y,
                               'effective_F': effF, 'first_stage': fs.loc['log_upwind_frp', 'Estimate'],
                               'iv_coef': iv['Estimate'], 'iv_se': iv['Std. Error'], 'iv_p': iv['Pr(>|t|)'],
                               'ar_p': arp, 'mean_y': d[y].mean(), 'n_districts': d['district'].nunique()})
            print(f"  {y:10s} IV {iv['Estimate']:+.4f} (se {iv['Std. Error']:.4f}, p={iv['Pr(>|t|)']:.3f}) | AR p={arp:.3f}")

res = pd.concat([res, pd.DataFrame(maiac_rows)], ignore_index=True)
res.to_csv(OUT_CSV, index=False)
print(f'\nSaved: {OUT_CSV} (now includes MAIAC rows)')

MAIAC panel: 59,893 of 60,125 district-months have valid AOD this month and last
Corr(log upwind FRP, log MAIAC AOD): 0.156



== log_maiac | baseline | full panel (481 districts) | effective F = 8.5 | first stage +0.0022 (p=0.456) ==


  events     IV +0.2591 (se 0.3177, p=0.421) | AR p=0.136


  pv_events  IV +0.1714 (se 0.0779, p=0.035) | AR p=0.026


  riots_vac  IV +0.1390 (se 0.0740, p=0.069) | AR p=0.043


  vac        IV +0.0567 (se 0.0270, p=0.044) | AR p=0.042

== log_maiac | baseline | active >= 30% (85 districts) | effective F = 4.2 | first stage +0.0047 (p=0.281) ==
  events     IV -2.9671 (se 1.9425, p=0.136) | AR p=0.005
  pv_events  IV -0.1760 (se 0.2875, p=0.545) | AR p=0.197


... (7 outputs left out)

  vac        IV +0.0463 (se 0.0333, p=0.174) | AR p=0.080

== log_maiac | + own fire | active >= 30% (85 districts) | effective F = 6.2 | first stage +0.0069 (p=0.083) ==
  events     IV -2.3304 (se 1.4816, p=0.126) | AR p=0.023
  pv_events  IV -0.1673 (se 0.2874, p=0.565) | AR p=0.221


  riots_vac  IV -0.0973 (se 0.2118, p=0.649) | AR p=0.099
  vac        IV -0.0666 (se 0.1613, p=0.683) | AR p=0.028

== log_maiac | + own fire | active >= 30%, 2018+ (85 districts) | effective F = 3.5 | first stage +0.0070 (p=0.095) ==
  events     IV -3.1329 (se 2.2447, p=0.172) | AR p=0.033


  pv_events  IV -0.5280 (se 0.7572, p=0.491) | AR p=0.109
  riots_vac  IV -0.2395 (se 0.4656, p=0.611) | AR p=0.055
  vac        IV -0.1945 (se 0.3783, p=0.611) | AR p=0.012

== log_maiac | + own fire | active >= 30%, no MERRA-2 grid point (60 districts) | effective F = 6.1 | first stage +0.0067 (p=0.103) ==


  events     IV -2.3438 (se 1.6729, p=0.171) | AR p=0.134
  pv_events  IV -0.0401 (se 0.2027, p=0.845) | AR p=0.374
  riots_vac  IV -0.0729 (se 0.2302, p=0.754) | AR p=0.230
  vac        IV -0.0466 (se 0.1471, p=0.754) | AR p=0.143

Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/analysis/output/pollution_iv_results.csv (now includes MAIAC rows)


### 8c. Why is the MAIAC first stage weak?

Diagnostics for the first stage of log MAIAC AOD on log upwind FRP (conflict-active and full samples): without the lagged-AOD control; restricted to district-months with higher retrieval coverage; and, for comparison, MERRA-2 log AOD and log PM2.5 on the same district-months. If smoke-heavy days are disproportionately missing from MAIAC (thick smoke flagged as cloud), the MAIAC response should be weaker in low-coverage months and the haze-season mean should be attenuated.

In [13]:
pmm = pm.merge(poll[['district', 'year', 'month']].assign(_ok=1), on=['district', 'year', 'month'])
diag = []
def fs_row(d, endog, ctrls, label):
    m = pf.feols(f'{endog} ~ {" + ".join(INSTR + ctrls)} | {FE}', data=d, vcov=VCOV).tidy().loc['log_upwind_frp']
    diag.append({'sample': label, 'endog': endog, 'lag_control': bool(ctrls), 'coef': m['Estimate'], 'p': m['Pr(>|t|)'], 'n': len(d)})
    print(f"  {label:38s} {endog:10s} lag ctrl={str(bool(ctrls)):5s} coef {m['Estimate']:+.4f} (p={m['Pr(>|t|)']:.3f}), n={len(d):,}")

for sample, d in [('full panel', pmm), ('active >= 30%', pmm[pmm['district'].isin(active)])]:
    print(f'\n== {sample} ==')
    for endog in ['log_maiac', 'log_aod', 'log_pm25']:
        fs_row(d, endog, [], sample)
        fs_row(d, endog, [f'{endog}_l1'], sample)
    for cut in [0.2, 0.4]:
        fs_row(d[d['maiac_coverage'] >= cut], 'log_maiac', [], f'{sample}, coverage >= {cut}')

c = pmm.groupby('month')[['maiac_coverage']].mean()
print('\nCorr(log upwind FRP, MAIAC coverage), full panel:',
      round(pmm['log_upwind_frp'].corr(pmm['maiac_coverage']), 3))
pd.DataFrame(diag).to_csv(f'{ROOT}/analysis/output/maiac_first_stage_diagnostics.csv', index=False)
print('Saved: analysis/output/maiac_first_stage_diagnostics.csv')


== full panel ==


  full panel                             log_maiac  lag ctrl=False coef -0.0025 (p=0.485), n=59,893
  full panel                             log_maiac  lag ctrl=True  coef +0.0022 (p=0.456), n=59,893


  full panel                             log_aod    lag ctrl=False coef +0.0169 (p=0.001), n=59,893


  full panel                             log_aod    lag ctrl=True  coef +0.0198 (p=0.000), n=59,893
  full panel                             log_pm25   lag ctrl=False coef +0.0283 (p=0.000), n=59,893
  full panel                             log_pm25   lag ctrl=True  coef +0.0274 (p=0.000), n=59,893
  full panel, coverage >= 0.2            log_maiac  lag ctrl=False coef +0.0084 (p=0.049), n=29,139


  full panel, coverage >= 0.4            log_maiac  lag ctrl=False coef +0.0082 (p=0.156), n=10,440

== active >= 30% ==
  active >= 30%                          log_maiac  lag ctrl=False coef +0.0013 (p=0.815), n=10,531
  active >= 30%                          log_maiac  lag ctrl=True  coef +0.0047 (p=0.281), n=10,531
  active >= 30%                          log_aod    lag ctrl=False coef +0.0101 (p=0.131), n=10,531
  active >= 30%                          log_aod    lag ctrl=True  coef +0.0159 (p=0.000), n=10,531
  active >= 30%                          log_pm25   lag ctrl=False coef +0.0204 (p=0.003), n=10,531
  active >= 30%                          log_pm25   lag ctrl=True  coef +0.0212 (p=0.000), n=10,531
  active >= 30%, coverage >= 0.2         log_maiac  lag ctrl=False coef +0.0075 (p=0.233), n=5,620
  active >= 30%, coverage >= 0.4         log_maiac  lag ctrl=False coef -0.0001 (p=0.982), n=2,303

Corr(log upwind FRP, MAIAC coverage), full panel: 0.241
Saved: analysis/output/m